현재까지는 workflow 기반의 agent를 구성하는것을 학습하였으며
이번부터는 agent가 자신이 사용할 tool을 직접 선별하는 agents 기반으로 구성한다

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o")

small_llm = ChatOpenAI(model="gpt-4o-mini")

In [4]:
from langchain_core.tools import tool


# tool 구성에는 세가지 구성이 필요하다
# 1. decorater (@tool)
# 2. description ("해당 툴의 설명")
# 3. expected arg ("인자")
@tool
def add(a: int, b: int) -> int:
    """숫자 a와 b를 더합니다."""
    return a+b

@tool
def multiply(a: int, b: int) -> int:
    """숫자 a와 b를 곱합니다."""
    return a*b


In [5]:
add.invoke({"a": 1, "b": 2})

3

In [6]:
llm_with_tools = small_llm.bind_tools([add, multiply])


In [7]:
query = "1과 2를 더하고 곱해서 출력하세요"

In [8]:
llm.invoke(query)

AIMessage(content='1과 2를 더하면 3이 됩니다. 1과 2를 곱하면 2가 됩니다. 따라서 더한 값은 3이고, 곱한 값은 2입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 19, 'total_tokens': 64, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_6b6e24b474', 'id': 'chatcmpl-BNK0jd87cGxb5GeB0Yp7j79HJDxVy', 'finish_reason': 'stop', 'logprobs': None}, id='run-9f7bb0a8-fe2a-4696-83c6-35a8ea63725f-0', usage_metadata={'input_tokens': 19, 'output_tokens': 45, 'total_tokens': 64, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [ ]:
result = llm_with_tools.invoke(query)

In [13]:
result.tool_calls

[{'name': 'add',
  'args': {'a': 1, 'b': 2},
  'id': 'call_fdnr7CpNj2G8KBLdP7M7gpHe',
  'type': 'tool_call'},
 {'name': 'multiply',
  'args': {'a': 1, 'b': 2},
  'id': 'call_4Xhpk43p34EMTgCsIOmKVWtL',
  'type': 'tool_call'}]

In [14]:
from langchain_core.messages import AnyMessage, HumanMessage
from typing import Sequence

message_list: Sequence[AnyMessage] = []
human_message = HumanMessage(query)
message_list.append(human_message)


In [15]:
# 현재 ai_message에는 human_message만 있는 상태
# ai_message, tool_message도 추가해주어야한다.
ai_message = llm_with_tools.invoke(message_list)

In [16]:
ai_message.tool_calls

[{'name': 'add',
  'args': {'a': 1, 'b': 2},
  'id': 'call_YRXSzeeSK6SY0tQNvIDVFEJC',
  'type': 'tool_call'},
 {'name': 'multiply',
  'args': {'a': 1, 'b': 2},
  'id': 'call_cm3uyuIkW1NdASO5OV4bFUTW',
  'type': 'tool_call'}]

In [17]:
# ai_message 추가
message_list.append(ai_message)

In [ ]:
# tool_message 추가
add_tool_message = add.invoke(ai_message.tool_calls[0])
multiply_tool_message = multiply.invoke(ai_message.tool_calls[1])

message_list.append(add_tool_message)
message_list.append(multiply_tool_message)









In [21]:
llm_with_tools.invoke(message_list)

AIMessage(content='1과 2를 더한 결과는 3이고, 곱한 결과는 2입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 154, 'total_tokens': 178, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0392822090', 'id': 'chatcmpl-BNK8W8iHi5bNF9AA59LsR1CneROo0', 'finish_reason': 'stop', 'logprobs': None}, id='run-5e291c9d-fdd4-404f-a077-65bf801f2245-0', usage_metadata={'input_tokens': 154, 'output_tokens': 24, 'total_tokens': 178, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})